# CSR Data Processing by Year
This notebook uses the functions from `process_csr_data.py` to extract, classify, and flatten year-over-year CSR datasets. Subsequent years can be added in new sections below.

In [1]:
import pandas as pd
from process_csr_data import (
    ensure_dirs,
    extract_year_label,
    step1_extract_and_save_locations,
    step2_geopy_classification,
    step3_heuristics_and_flatten
)

# Ensure output directories exist (location/, location_geopy/, flattened/)
ensure_dirs()

# Global paths
master_mirror_path = 'Location_Classification_Mirror.csv'

# Year: 2015-16
Configure the inputs and run the 3-step pipeline for the `CSR_activities_2015-16.xlsx` dataset.

In [2]:
input_file_15_16 = 'CSR_activities_2015-16.xlsx'
year_label_15_16 = extract_year_label(input_file_15_16)

print(f"Targeting file: {input_file_15_16}")
print(f"Extracted Year Label: {year_label_15_16}")

Targeting file: CSR_activities_2015-16.xlsx
Extracted Year Label: 2015-16


## Step 1: Extract Locations
Parses `States (City/Town/District/Village)` and dumps unique raw locations to the `location/` folder.

In [3]:
raw_df_15_16, extracted_locs_df_15_16 = step1_extract_and_save_locations(input_file_15_16, year_label_15_16)
if extracted_locs_df_15_16 is not None:
    display(extracted_locs_df_15_16.head())


--- STEP 1: Extraction for 2015-16 ---
Extracted 1537 unique locations to location/Location_Classification_2015-16.csv


,Location,Class
0,ALL STATES,State
1,ANDAMAN & NICOBAR ISLANDS,State
2,ANDHRA PRADESH,State
3,ARUNACHAL PRADESH,State
4,ASSAM,State


## Step 2: Geopy Classification
Checks against the master mirror map. Dispatches API requests only for brand new locations. Updates global mirror map and saves local geocodes to `location_geopy/`.

In [4]:
if extracted_locs_df_15_16 is not None and not extracted_locs_df_15_16.empty:
    updated_master_15_16 = step2_geopy_classification(extracted_locs_df_15_16, master_mirror_path, year_label_15_16)
    display(updated_master_15_16.tail())
else:
    print("No valid extracted locations found. Skipping Step 2.")


--- STEP 2: Geopy Classification for 2015-16 ---
Loaded master mirror dataset with 2643 mapped locations.
Executed Geopy queries for 0 new locations.
Saved 2015-16 geopy output to location_geopy/Location_Geopy_2015-16.csv
Updated Master Mirror dataset at Location_Classification_Mirror.csv


,Location,Intelligent_Class
2638,WORLI,Town
2639,YALAHANKA,Village
2640,YERAWADA,Town
2641,YEVATMAL,Town
2642,ZIRAKPUR,Town


## Step 3: Heuristics and Flatten
Resolves any unmapped elements using Indian heuristic keywords, then dynamically expands the original `xlsx` mapping rows into entirely flat dimensional entities in the `flattened/` folder.

In [5]:
if extracted_locs_df_15_16 is not None and not extracted_locs_df_15_16.empty:
    flattened_df_15_16 = step3_heuristics_and_flatten(raw_df_15_16, updated_master_15_16, master_mirror_path, year_label_15_16)
    
    # Preview the unraveled columns
    display(flattened_df_15_16.head(10))
else:
    print("No valid extracted locations found. Skipping Step 3.")


--- STEP 3: Heuristics & Flattening for 2015-16 ---
Flattening complete! Saved 12853 records to flattened/CSR_activities_Unravelled_2015-16.csv


,Symbol,Company,Activity,Sector,Schedule,Amount to be Spent as decided by Company (Rs.lacs),Actual Amount Spent (Rs.lacs),Direct-Agency Name/s,Direct-Amount (Rs.lacs),Implementing Agency Name/s,Implementing Agency-Amount (Rs.lacs),Resolved_State,Resolved_Location,Location_Class
0,20MICRONS,20 MICRONS LTD.,HEALTH CARE AND MEDICAL FACILITIES,NaN,SCHEDULE VII (I),1.85,1.85,NaN,NaN,NaN,1.850,GUJARAT,VADODARA,City
1,3MINDIA,3M INDIA LTD.,EDUCATION SUPPORT TWO (2) MOBILE SCIENCE LABS ...,NaN,SCHEDULE VII (II),61.70,42.05,NaN,NaN,"UNITED WAY OF BENGALURU, GOVT.OF TAMIL NADU",42.053,KARNATAKA,ANEKAL,City
2,3MINDIA,3M INDIA LTD.,EDUCATION SUPPORT TWO (2) MOBILE SCIENCE LABS ...,NaN,SCHEDULE VII (II),61.70,42.05,NaN,NaN,"UNITED WAY OF BENGALURU, GOVT.OF TAMIL NADU",42.053,KARNATAKA,BENGALURU,City
3,3MINDIA,3M INDIA LTD.,EDUCATION SUPPORT TWO (2) MOBILE SCIENCE LABS ...,NaN,SCHEDULE VII (II),61.70,42.05,NaN,NaN,"UNITED WAY OF BENGALURU, GOVT.OF TAMIL NADU",42.053,MAHARASHTRA,PUNE,City
4,3MINDIA,3M INDIA LTD.,EDUCATION SUPPORT TWO (2) MOBILE SCIENCE LABS ...,NaN,SCHEDULE VII (II),61.70,42.05,NaN,NaN,"UNITED WAY OF BENGALURU, GOVT.OF TAMIL NADU",42.053,MAHARASHTRA,RANJANGAON,Village
5,3MINDIA,3M INDIA LTD.,NATIONAL CALAMITY DISASTER RELIEF SUPPORT DURI...,NaN,SCHEDULE VII (VIII),51.68,13.13,NaN,NaN,"UNITED WAY OF BENGALURU, GOVT.OF TAMIL NADU",13.125,TAMIL NADU,CHENNAI,City
6,3MINDIA,3M INDIA LTD.,SOCIAL INNOVATION SUPPORT AN INCUBATION FUND P...,NaN,SCHEDULE VII (IX),51.40,9.84,NaN,NaN,"UNITED WAY OF BENGALURU, GOVT.OF TAMIL NADU",9.841,ALL STATES,ALL STATES,State
7,3MINDIA,3M INDIA LTD.,WOMEN EMPOWERMENT SUPPORT TWO(2) SKILLS DEVELO...,NaN,SCHEDULE VII (III),41.10,22.95,NaN,NaN,"UNITED WAY OF BENGALURU, GOVT.OF TAMIL NADU",22.950,KARNATAKA,BENGALURU,City
8,63MOONS,63 MOONS TECHNOLOGIES LTD.,PROMOTING EDUCATION,NaN,SCHEDULE VII (II),30.00,30.00,NaN,NaN,SRIMATI SAVITA MAHILA UTKARSH SANGH,30.000,GUJARAT,AHMEDABAD,City
9,63MOONS,63 MOONS TECHNOLOGIES LTD.,PROMOTING EDUCATION,NaN,SCHEDULE VII (II),35.33,35.33,NaN,NaN,VIDYA PRASARAK MANDAL,35.330,MAHARASHTRA,THANE,District


# Year: 2016-17
Configure the inputs and run the 3-step pipeline for the `CSR_activities_2016-17.xlsx` dataset.

In [6]:
input_file_16_17 = 'CSR_activities_2016-17.xlsx'
year_label_16_17 = extract_year_label(input_file_16_17)

print(f"Targeting file: {input_file_16_17}")
print(f"Extracted Year Label: {year_label_16_17}")

Targeting file: CSR_activities_2016-17.xlsx
Extracted Year Label: 2016-17


## Step 1: Extract Locations
Parses `States (City/Town/District/Village)` and dumps unique raw locations to the `location/` folder.

In [7]:
raw_df_16_17, extracted_locs_df_16_17 = step1_extract_and_save_locations(input_file_16_17, year_label_16_17)
if extracted_locs_df_16_17 is not None:
    display(extracted_locs_df_16_17.head())


--- STEP 1: Extraction for 2016-17 ---
Extracted 1630 unique locations to location/Location_Classification_2016-17.csv


,Location,Class
0,),State
1,ALL STATES,State
2,ANDAMAN & NICOBAR ISLANDS,State
3,ANDHRA PRADESH,State
4,ARUNACHAL PRADESH,State


## Step 2: Geopy Classification
Checks against the master mirror map. Dispatches API requests only for brand new locations. Updates global mirror map and saves local geocodes to `location_geopy/`.

In [9]:
if extracted_locs_df_16_17 is not None and not extracted_locs_df_16_17.empty:
    updated_master_16_17 = step2_geopy_classification(extracted_locs_df_16_17, master_mirror_path, year_label_16_17)
    display(updated_master_16_17.tail())
else:
    print("No valid extracted locations found. Skipping Step 2.")


--- STEP 2: Geopy Classification for 2016-17 ---
Loaded master mirror dataset with 2643 mapped locations.
Executed Geopy queries for 422 new locations.
Saved 2016-17 geopy output to location_geopy/Location_Geopy_2016-17.csv
Updated Master Mirror dataset at Location_Classification_Mirror.csv


,Location,Intelligent_Class
3117,VULLI,Town
3118,WARANGAL URBAN DIST.,District
3119,WEST BENGAL (8 DIST.S,District
3120,YADAGIRIGUTTA MANDAL,District
3121,YEDULA,Village


## Step 3: Heuristics and Flatten
Resolves any unmapped elements using Indian heuristic keywords, then dynamically expands the original `xlsx` mapping rows into entirely flat dimensional entities in the `flattened/` folder.

In [10]:
if extracted_locs_df_16_17 is not None and not extracted_locs_df_16_17.empty:
    flattened_df_16_17 = step3_heuristics_and_flatten(raw_df_16_17, updated_master_16_17, master_mirror_path, year_label_16_17)
    
    # Preview the unraveled columns
    display(flattened_df_16_17.head(10))
else:
    print("No valid extracted locations found. Skipping Step 3.")


--- STEP 3: Heuristics & Flattening for 2016-17 ---
Applied Heuristics globally. 'Not Found' reduced from 86 to 0.
Master mirror updated.
Flattening complete! Saved 14279 records to flattened/CSR_activities_Unravelled_2016-17.csv


,Symbol,Company,Activity,Sector,Schedule,Amount to be Spent as decided by Company (Rs.lacs),Actual Amount Spent (Rs.lacs),Direct-Agency Name/s,Direct-Amount (Rs.lacs),Implementing Agency Name/s,Implementing Agency-Amount (Rs.lacs),Resolved_State,Resolved_Location,Location_Class
0,20MICRONS,20 MICRONS LTD.,HEALTH CARE AND MEDICAL FACILITIES,NaN,SCHEDULE VII (I),5.1,5.10,20 MICRONS FOUNDATION TRUST,2.550,NaN,2.550,GUJARAT,VADODARA,City
1,3MINDIA,3M INDIA LTD.,ASH INITIATIVE TO BUILD SANITATION FACILITIES ...,NaN,SCHEDULE VII (I),108.0,42.50,NaN,NaN,"UNITED WAY OF BENGALURU, GOVT.OF TAMIL NADU",42.500,MAHARASHTRA,PUNE,City
2,3MINDIA,3M INDIA LTD.,ASH INITIATIVE TO BUILD SANITATION FACILITIES ...,NaN,SCHEDULE VII (I),108.0,42.50,NaN,NaN,"UNITED WAY OF BENGALURU, GOVT.OF TAMIL NADU",42.500,MAHARASHTRA,RANJANGAON,Village
3,3MINDIA,3M INDIA LTD.,EDUCATION,NaN,SCHEDULE VII (II),108.0,198.67,CHRISTEL HOUSE,99.335,"UNITED WAY OF BENGALURU, GOVT.OF TAMIL NADU",99.335,KARNATAKA,ANEKAL,City
4,3MINDIA,3M INDIA LTD.,EDUCATION,NaN,SCHEDULE VII (II),108.0,198.67,CHRISTEL HOUSE,99.335,"UNITED WAY OF BENGALURU, GOVT.OF TAMIL NADU",99.335,KARNATAKA,BENGALURU,City
5,3MINDIA,3M INDIA LTD.,EDUCATION,NaN,SCHEDULE VII (II),108.0,198.67,CHRISTEL HOUSE,99.335,"UNITED WAY OF BENGALURU, GOVT.OF TAMIL NADU",99.335,MAHARASHTRA,PUNE,City
6,3MINDIA,3M INDIA LTD.,EDUCATION,NaN,SCHEDULE VII (II),108.0,198.67,CHRISTEL HOUSE,99.335,"UNITED WAY OF BENGALURU, GOVT.OF TAMIL NADU",99.335,MAHARASHTRA,RANJANGAON,Village
7,3MINDIA,3M INDIA LTD.,EDUCATION,NaN,SCHEDULE VII (II),108.0,198.67,CHRISTEL HOUSE,99.335,"UNITED WAY OF BENGALURU, GOVT.OF TAMIL NADU",99.335,WEST BENGAL,KOLKATA,City
8,3MINDIA,3M INDIA LTD.,SOCIAL INNOVATION,NaN,SCHEDULE VII (IX),36.0,19.41,NaN,NaN,"UNITED WAY OF BENGALURU, GOVT.OF TAMIL NADU",19.410,ALL STATES,ALL STATES,State
9,3MINDIA,3M INDIA LTD.,WOMEN EMPOWERMENT,NaN,SCHEDULE VII (III),72.0,86.00,NaN,NaN,"UNITED WAY OF BENGALURU, GOVT.OF TAMIL NADU",86.000,KARNATAKA,BENGALURU,City
